In [ ]:
import sys
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-colorblind')


In [ ]:
import yaml

base_path = Path('../../../').resolve()
sys.path.append(str(base_path))
from helpers import singlecell_utils

with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
NCPUS = 40
PAS_STRATIFICATION = 'ageXclass'
PAS_DIR = Path(f'{base_path}run_SCAPTURE/2_filter_pas/pl_{PAS_STRATIFICATION}/filtered_pas/')
BED_SUFFIX = '.filtered_pas.bed'

In [ ]:
columns = ['chrom', 'start', 'end', 'name', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 
           'blockCount', 'blockSizes', 'blockStarts',]

# Read one PAS file

In [ ]:
cell_type = 'Young_adult-EN'
pas_passed_path = PAS_DIR / f'{cell_type}.filtered_pas.bed'

df_peaks = pd.read_table(pas_passed_path, index_col=False, header=None, names=columns)
df_peaks

In [ ]:
se_name_split = df_peaks.name.str.split('|')

df_peaks['geneName'] = se_name_split.apply(lambda x: x[0])
df_peaks['geneClass'] = se_name_split.apply(lambda x: x[3])
df_peaks['transcriptName'] = se_name_split.apply(lambda x: x[4])
df_peaks['region'] = se_name_split.apply(lambda x: x[5])

In [ ]:
# In case of intron or 3p ext, transcript name is empty. Fill them with gene name
df_peaks['transcriptName'] = df_peaks['transcriptName'].where(df_peaks['transcriptName'] != 'null', df_peaks['geneName'])
df_peaks[['name', 'geneName', 'transcriptName', 'region', 'geneClass']]

# Check how many will be removed

In [ ]:
peak_identifiers = 'chrom start end strand blockSizes blockStarts'.split()

In [ ]:
df_peaks_uniq = df_peaks.groupby(peak_identifiers, observed=True).first().reset_index()
len(df_peaks_uniq)

In [ ]:
# This amount of peaks are removed
removed_pct = 100 *(len(df_peaks) - len(df_peaks_uniq)) / len(df_peaks)
print(f'This amount of PAS came from the same peak, so will be removed: {removed_pct:.2f}%')

In [ ]:
samepas_counts = Counter(df_peaks.groupby(peak_identifiers, observed=True).name.count())
samepas_counts

In [ ]:
samepas_counts_less3 = {'≥4': 0}

for k, v in samepas_counts.items():
    if k == 1: continue
    if k >= 4:
        samepas_counts_less3['≥4'] += v
    else:
        samepas_counts_less3[k] = v
        
samepas_counts_less3

In [ ]:
df = pd.DataFrame(samepas_counts_less3, index=['Before overlap removal'])
df = df[[2, 3, '≥4']]
df.plot(kind='barh', stacked=True)
plt.show()

df

# Add gene length from reference gtf 

In [ ]:
GTF_PATH = '/sc/arion/projects/CommonMind/yeon/p/APA/ref/ensembl.104/Homo_sapiens.GRCh38.104.filt_renamed.gtf'

In [ ]:
df_gtf = pd.read_table(GTF_PATH, header=None, index_col=False, skiprows=10, comment='#', dtype={0: 'category'})
df_gtf.head()

In [ ]:
len(df_gtf)
df_gtf = df_gtf[df_gtf[2]=='gene']
len(df_gtf)

In [ ]:
# get gene length
def parse_desc(desc, second_delim):
    return dict(field.strip().replace('"', '').split(second_delim) for field in desc.removesuffix(';').split(';'))
    
df_gtf_desc = pd.DataFrame(list(df_gtf[8].apply(parse_desc, args=' ')),
                           index=df_gtf.index).convert_dtypes()

In [ ]:
se_genelen = df_gtf[4] - df_gtf[3]

genename_to_length = dict(zip(df_gtf_desc['gene_name'], se_genelen))
geneid_to_length = dict(zip(df_gtf_desc['gene_id'], se_genelen))

In [ ]:
def map_gene_length(gn):
    if gn in genename_to_length:
        return genename_to_length[gn]
        
    elif gn in geneid_to_length:
        return geneid_to_length[gn]

    else:
        return 0.0

df_peaks['geneLength'] = df_peaks['geneName'].apply(map_gene_length)

# Add PsychAD read counts per subclass, and define class - subclass mapping

In [ ]:
# Gene metadata required
FULL_H5AD_PATH = '/sc/arion/projects/psychAD/NPS-AD/freeze2_proc/240124_PsychAD_freeze3_FULL_clean.h5ad'
psyad_full = singlecell_utils.read_everything_but_X(FULL_H5AD_PATH)

In [ ]:
psyad_full.var.head()

In [ ]:
df_psb = pd.read_pickle('../PsychAD_age_X_class_psb_cnt_cpm.pkl')
df_psb.head()

In [ ]:
# PSB and var has the same index

(df_psb.index == psyad_full.var.index).all()

In [ ]:
psycpm_cols = [col for col in df_psb.columns if col.startswith('cpm_')]

In [ ]:
df_psyad_gene = psyad_full.var

for ct in psycpm_cols:
    df_psyad_gene[ct] = df_psb[ct]

In [ ]:
df_psyad_gene.columns

In [ ]:
# {'class_Astro': {'DROSHA': 101.1, }}

celltype_genename_to_cpm = dict([(ct, dict(zip(df_psyad_gene['gene_name'], df_psyad_gene[ct]))) for ct in psycpm_cols])
celltype_geneid_to_cpm = dict([(ct, dict(zip(df_psyad_gene['gene_id'], df_psyad_gene[ct]))) for ct in psycpm_cols])


In [ ]:
print(Counter(df_peaks['geneName'].isin(df_psyad_gene.gene_name)))
print(Counter(df_peaks['geneName'].isin(df_psyad_gene.gene_id)))
print(Counter(df_peaks['geneName'].isin(df_psyad_gene.gene_name) | df_peaks['geneName'].isin(df_psyad_gene.gene_id)))

In [ ]:
# Mostly pseudogenes
df_peaks[~(df_peaks['geneName'].isin(df_psyad_gene.gene_name) | df_peaks['geneName'].isin(df_psyad_gene.gene_id))]

In [ ]:
def map_psychad_cpm(gn, ct):
    genename_to_cpm = celltype_geneid_to_cpm[ct]
    geneid_to_cpm = celltype_genename_to_cpm[ct]
    
    if gn in genename_to_cpm:
        return genename_to_cpm[gn]
        
    elif gn in geneid_to_cpm:
        return geneid_to_cpm[gn]

    else:
        return 0.0

df_peaks['psychAdCpm'] = df_peaks['geneName'].apply(map_psychad_cpm, args=['cpm_' + cell_type])

In [ ]:
df_peaks

In [ ]:
# Free some memory
del psyad_full

# Add overlapped long-read count

In [ ]:
# Column 
df_lr = pd.read_pickle('../long-read_3p_and_psb_cnt_cpm.pkl')
df_lr

## Define cell-type mapper

In [ ]:
bed_paths = sorted(PAS_DIR.glob(f'*{BED_SUFFIX}'))
cell_types = [str(pt.name).removesuffix(BED_SUFFIX) for pt in bed_paths]
cell_types

In [ ]:
celltype_to_longreadtype = dict([(ct, ct.split('-')[1]) for ct in cell_types])
celltype_to_longreadtype

In [ ]:
# 3p end position of each peak

def get_peak_3p_end(se_row):
    if se_row.strand=='+':
        return se_row.end - 1
    else:
        return se_row.start

df_peaks['3pEnd'] = df_peaks.apply(get_peak_3p_end, axis=1)
df_peaks

In [ ]:
# If same name and close 3p end, PAS = long-read transcript
def find_longread(se_row, hashed_3p_counts, lr_cell_type, margin=50):
    FARAWAY = 987654321
    key = se_row.chrom, se_row.strand, se_row.geneName
    
    if key in hashed_3p_counts:
        peak_3p = se_row['3pEnd']
        df = hashed_3p_counts[key]

        start_3p_col = f'start-{margin}'
        end_3p_col = f'end+{margin}'

        se_overlap = (df[start_3p_col] <= peak_3p) & (peak_3p < df[end_3p_col])
        total_cpm = df[se_overlap][lr_cell_type].sum()
        min_dist = (df.start - peak_3p).abs().min()
        if min_dist > margin: # If faraway, distance is not useful.
            min_dist = FARAWAY

        return total_cpm, min_dist
    
    else:
        return 0.0, FARAWAY

    
hashed_3p_counts = dict(iter(df_lr.groupby(['chr', 'strand', 'gene_name'])))
lr_cell_type = celltype_to_longreadtype[cell_type]
find_longread(df_peaks.loc[0], hashed_3p_counts, lr_cell_type)

In [ ]:
def apply_runner(data, func_applied, args=None, axis=None):
    if axis:
        return data.apply(func_applied, args=args, axis=axis)
    return data.apply(func_applied, args=args)

def parallel_apply(data, func_applied, lArgs=None, nWorkers=NCPUS, axis=None):
    lChunks = np.array_split(data, nWorkers)
    lWorkers = []
    with ProcessPoolExecutor(max_workers=nWorkers) as executor:
        for i in range(nWorkers):
            future = executor.submit(apply_runner, lChunks[i], func_applied, args=lArgs, axis=axis)
            lWorkers.append(future)
            
    return pd.concat([fut.result() for fut in lWorkers])

se_count_and_dist = parallel_apply(df_peaks, find_longread, lArgs=[hashed_3p_counts, lr_cell_type], axis=1)

In [ ]:
df_peaks['longReadCpm'] = se_count_and_dist.apply(lambda x: x[0])
df_peaks['longReadDist'] = se_count_and_dist.apply(lambda x: x[1])

In [ ]:

fig = plt.figure()
ax = fig.add_subplot(111)
df_peaks.longReadDist.hist(ax=ax)
ax.set_title(f'SCAPTURE vs long-read ({cell_type})')
ax.set_xlabel('Distance (bp)')
ax.set_ylabel('Number of PAS peaks')
plt.show()

plt.show()

# Add regional priorities

In [ ]:
region_priority = '3UTR CDS exon intron 3primeExtended 5UTR'.split()
region_priority = dict([(y, x) for x, y in enumerate(region_priority)])
region_priority

In [ ]:
df_peaks['regionPriority'] = df_peaks.region.map(region_priority)
df_peaks.head()

# Filter the same PAS peaks

In [ ]:
# Sort and keep first not works, since we need ties also.
def filter_max_or_tie(df, field):
    if len(df)==1:
        return df
    choosen = df[field].max()
    return df[df[field]==choosen]

def filter_min_or_tie(df, field):
    if len(df)==1:
        return df
        
    choosen = df[field].min()
    return df[df[field]==choosen]
    

In [ ]:
df_peaks

## 1. Sort by PsychAD final CPM

In [ ]:
df_peaks_psycpm = df_peaks.groupby(peak_identifiers, observed=True).apply(
                                    lambda x: filter_max_or_tie(x, 'psychAdCpm'), include_groups=False).reset_index()
del df_peaks_psycpm[f'level_{len(peak_identifiers)}']
len(df_peaks_psycpm), len(df_peaks), len(df_peaks_uniq)

In [ ]:
# If still duplicated, read count is 0
df_peaks[df_peaks.duplicated(peak_identifiers + ['psychAdCpm'], keep=False)]['psychAdCpm'].describe()

## 2. Sort by long-read count 

In [ ]:
# Choose maximal count rows
df_peaks_lrcpm = df_peaks_psycpm.groupby(peak_identifiers, observed=True).apply(
                                          lambda x: filter_max_or_tie(x, 'longReadCpm'),include_groups=False).reset_index()
del df_peaks_lrcpm[f'level_{len(peak_identifiers)}']

len(df_peaks_lrcpm), len(df_peaks_psycpm), len(df_peaks), len(df_peaks_uniq)

## 3. Sort by regional type (exon, intron...)

In [ ]:
df_peaks_rgn = df_peaks_lrcpm.groupby(peak_identifiers, observed=True).apply(
                                      lambda x: filter_min_or_tie(x, 'regionPriority'), include_groups=False).reset_index()
del df_peaks_rgn[f'level_{len(peak_identifiers)}']
len(df_peaks_rgn), len(df_peaks_lrcpm), len(df_peaks_psycpm), len(df_peaks), len(df_peaks_uniq)

## 4. Sort by gene length

In [ ]:
df_peaks_genelen = df_peaks_rgn.groupby(peak_identifiers, observed=True).apply(
                                           lambda x: filter_max_or_tie(x, 'geneLength'), include_groups=False).reset_index()
del df_peaks_genelen[f'level_{len(peak_identifiers)}']
len(df_peaks_genelen), len(df_peaks_psycpm), len(df_peaks_rgn), len(df_peaks_lrcpm), len(df_peaks), len(df_peaks_uniq)

# Draw figure showing overlap removal

In [ ]:
df_overlapped_counts = pd.DataFrame([
    Counter(df_peaks_genelen.groupby(peak_identifiers, observed=True).name.count()),
    Counter(df_peaks_rgn.groupby(peak_identifiers, observed=True).name.count()),
    Counter(df_peaks_lrcpm.groupby(peak_identifiers, observed=True).name.count()),
    Counter(df_peaks_psycpm.groupby(peak_identifiers, observed=True).name.count()),
    Counter(df_peaks.groupby(peak_identifiers, observed=True).name.count()),
], 
index=[
    'After gene length',
    'After regional priority (exon > intron...)',
    'After long-read count',
    'After PsychAD full read count',
    'Before removal', 
]).fillna(0).astype(int)

df_overlapped_counts['≥4'] = df_overlapped_counts[df_overlapped_counts.columns[df_overlapped_counts.columns>=4]].sum(axis=1)
df_overlapped_counts

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2))

df_overlapped_counts[[2, 3, '≥4']].plot(kind='barh', stacked=True, ax=ax, zorder=10, width=0.7)

ax.legend(loc=(1.01, 0))
ax.set_xticks([0, 100, 200, 300, 400, 500])
ax.xaxis.grid()

ax.set_title(f'Duplicated PAS in {cell_type}\n({len(df_peaks):,} PAS in total)')

plt.show()


# Do the same thing for all PASs

In [ ]:
bed_paths = sorted(PAS_DIR.glob(f'*{BED_SUFFIX}'))
print(len(bed_paths))
bed_paths[0]

In [ ]:
def draw_stackbars(cell_type, df, df_psycpm, df_lrcpm, df_rgn, df_genelen):
    df_overlapped_counts = pd.DataFrame([
        Counter(df_genelen.groupby(peak_identifiers, observed=True).name.count()),
        Counter(df_rgn.groupby(peak_identifiers, observed=True).name.count()),
        Counter(df_lrcpm.groupby(peak_identifiers, observed=True).name.count()),
        Counter(df_psycpm.groupby(peak_identifiers, observed=True).name.count()),
        Counter(df.groupby(peak_identifiers, observed=True).name.count()),
    ], 
    index=[
        'After gene length',
        'After regional priority (exon > intron...)',
        'After long-read count',
        'After PsychAD read count',
        'Before removal', 
    ]).fillna(0).astype(int)

    df_overlapped_counts['≥4'] = df_overlapped_counts[df_overlapped_counts.columns[df_overlapped_counts.columns>=4]].sum(axis=1)

    if 2 not in df_overlapped_counts:
        df_overlapped_counts[2] = 0
    if 3 not in df_overlapped_counts:
        df_overlapped_counts[3] = 0
    
    fig, ax = plt.subplots(figsize=(3, 2))
    
    df_overlapped_counts[[2, 3, '≥4']].plot(kind='barh', stacked=True, ax=ax, zorder=10, width=0.7)
    
    ax.legend(loc=(1.01, 0))
    ax.set_xticks([0, 100, 200, 300, 400, 500])
    ax.xaxis.grid()
    
    ax.set_title(f'Duplicated PAS in {cell_type}\n({len(df):,} PAS in total)')

    plt.savefig(f'./pdf/dup_gene_pas_stat_{cell_type}.pdf', bbox_inches='tight')
    plt.show()


In [ ]:
def drop_duplicate_by_col(df, col, max=False):
    if max:
        df_ret = df.groupby(peak_identifiers, observed=True).apply(
            lambda x: filter_max_or_tie(x, col), include_groups=False).reset_index()
    else:
        df_ret = df.groupby(peak_identifiers, observed=True).apply(
            lambda x: filter_min_or_tie(x, col), include_groups=False).reset_index()
        
    del df_ret[f'level_{len(peak_identifiers)}']
    
    return df_ret

def process_single_cell_type(bed_path):
    cell_type = str(bed_path.name).removesuffix(BED_SUFFIX)
    
    df_peaks = pd.read_table(bed_path, index_col=False, header=None, names=columns)
    df_peaks['cellType'] = cell_type

    se_name_split = df_peaks.name.str.split('|')

    df_peaks['geneName'] = se_name_split.apply(lambda x: x[0])
    df_peaks['geneClass'] = se_name_split.apply(lambda x: x[3])
    df_peaks['region'] = se_name_split.apply(lambda x: x[5])

    df_peaks['geneLength'] = df_peaks['geneName'].apply(map_gene_length)
    df_peaks['psychAdCpm'] = df_peaks['geneName'].apply(map_psychad_cpm, args=['cpm_' + cell_type])
    
    df_peaks['3pEnd'] = df_peaks.apply(get_peak_3p_end, axis=1)
    lr_cell_type = celltype_to_longreadtype[cell_type]
    se_count_and_dist = df_peaks.apply(find_longread, args=[hashed_3p_counts, lr_cell_type], axis=1)
    df_peaks['longReadCpm'] = se_count_and_dist.apply(lambda x: x[0])
    df_peaks['longReadDist'] = se_count_and_dist.apply(lambda x: x[1])

    df_peaks['regionPriority'] = df_peaks.region.map(region_priority)

    # print function is not thread-safe. but I don't care.
    print(f'Processing {cell_type} with {len(df_peaks):,} lines...')

    # Choose more reads in PsychAD freeze3
    df_peaks_psycpm = drop_duplicate_by_col(df_peaks, 'psychAdCpm', max=True)
    
    # Choose maximal count rows
    df_peaks_lrcpm = drop_duplicate_by_col(df_peaks_psycpm, 'longReadCpm', max=True)
    
    # Choose exon > intron > 3p ext
    df_peaks_rgn = drop_duplicate_by_col(df_peaks_lrcpm, 'regionPriority', max=False)

    # Choose gene length
    df_peaks_genelen = drop_duplicate_by_col(df_peaks_rgn, 'geneLength', max=True)

    # write bed file
    out_path = OUT_DIR + bed_path.stem + '.rmdupgene.bed'
    df_peaks_genelen[columns].to_csv(out_path, sep='\t', index=False, header=False)

    # Draw figure
    draw_stackbars(cell_type, df_peaks, df_peaks_psycpm, df_peaks_lrcpm, df_peaks_rgn, df_peaks_genelen)

    # return all PAS with dup removal info
    df_sorted = df_peaks.sort_values(by='psychAdCpm longReadCpm regionPriority geneLength'.split(), 
                                     ascending=[False, False, True, False])
    #df_ret = df_sorted[df_sorted.duplicated(peak_identifiers, keep=False)].sort_values(peak_identifiers)

    return df_sorted
    

OUT_DIR = './dup_gene_removed/'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

PDF_DIR = './pdf/'
Path(PDF_DIR).mkdir(parents=True, exist_ok=True)

#for bed_path in bed_paths:
#    process_single_cell_type(bed_path)
    
workers = []
with ProcessPoolExecutor(max_workers=NCPUS) as executor:
    for bed_path in bed_paths:
        workers.append(executor.submit(process_single_cell_type, bed_path))
        
df_dup = pd.concat([wok.result() for wok in workers])

# Sort and save duplicated PASs

In [ ]:
df_sorted = df_dup.sort_values(by='psychAdCpm longReadCpm regionPriority geneLength'.split(), 
                                 ascending=[False, False, True, False])
df_sorted.sort_values(peak_identifiers, inplace=True)

# For Tsv, just merge everything to one file
df_sorted.to_pickle(f'PAS_duplicated_from_many_genes_removal_info-{PAS_STRATIFICATION}.pkl')
df_sorted.to_csv(f'PAS_duplicated_from_many_genes_removal_info-{PAS_STRATIFICATION}.tsv.gz', sep='\t', compression='gzip')


# Manually check removed genes

In [ ]:
cell_type = cell_types[0]

df_chck = df_sorted[df_sorted['cellType']==cell_type]
df_chck

In [ ]:
def check_priority(df):
    se_selected = df.iloc[0]
    max_psycpm = ((se_selected['psychAdCpm'] - df['psychAdCpm'].iloc[1:]) > 0.0).all()
    max_lrcpm = ((se_selected['longReadCpm'] - df['longReadCpm'].iloc[1:]) > 0.0).all()
    max_rgn = ((se_selected['regionPriority'] - df['regionPriority'].iloc[1:]) < 0.0).all()
    max_len = ((se_selected['geneLength'] - df['geneLength'].iloc[1:] > 0.1)).all()
    return [max_psycpm, max_lrcpm, max_rgn, max_len]
    
se_checked = df_chck.groupby(peak_identifiers, observed=True).apply(check_priority)

In [ ]:
se_checked[~se_checked.map(any)]

In [ ]:
se_checked[~se_checked.map(lambda x: x[0])]